# Initial Qwen3-4B GSM8K baseline experiment

Run the complete Week 1 pipeline on a deterministic 75-problem development subset. Use a GPU Colab runtime. Results are streamed to JSONL after every problem.

In [ ]:
from pathlib import Path
import os

REPOSITORY_URL = "https://github.com/Rami2212/PSSN-LLM-Reasoning.git"
REPOSITORY_DIR = Path("/content/PSSN-LLM-Reasoning")
if not (REPOSITORY_DIR / "requirements.txt").exists():
    !git clone {REPOSITORY_URL} {REPOSITORY_DIR}
os.chdir(REPOSITORY_DIR)
%pip install -q -r requirements.txt

In [ ]:
import json
from src.utils.environment import collect_environment_info

SUBSET_SIZE = 75
RANDOM_SEED = 42
MAX_NEW_TOKENS = 1024
RESULTS_PATH = Path("results/baseline/baseline_results.jsonl")
SUMMARY_PATH = Path("results/baseline/baseline_summary.json")
environment = collect_environment_info()
print(json.dumps(environment, indent=2))
assert environment["cuda_available"], "Select a GPU runtime before continuing."

In [ ]:
from src.data.gsm8k import load_gsm8k

gsm8k = load_gsm8k(development_size=SUBSET_SIZE, seed=RANDOM_SEED)
development = gsm8k["development"]
assert len(development) == SUBSET_SIZE
assert all(row["id"].startswith("gsm8k-train-") for row in development)
print(f"Prepared {len(development)} deterministic development problems.")

In [ ]:
from src.models.qwen import GenerationSettings, QwenGenerator

settings = GenerationSettings(
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=False,
    seed=RANDOM_SEED,
)
generator = QwenGenerator.from_pretrained(settings=settings)
print(f"Loaded {generator.model_id} using {generator.precision}.")

In [ ]:
from src.baseline.experiment import run_baseline_experiment
from src.utils import ExperimentLogger

logger = ExperimentLogger(
    RESULTS_PATH,
    experiment_name="qwen3-4b-gsm8k-cot-baseline",
    run_metadata={
        "dataset": "openai/gsm8k",
        "dataset_config": "main",
        "subset_source": "train",
        "subset_size": SUBSET_SIZE,
        "seed": RANDOM_SEED,
    },
)

def report_progress(index, record):
    print(
        f"[{index:02d}/{SUBSET_SIZE}] {record['problem_id']} | "
        f"correct={record['is_correct']} | tokens={record['output_tokens']} | "
        f"latency={record['inference_latency_seconds']:.2f}s"
    )

records = run_baseline_experiment(
    development, generator, logger, progress_callback=report_progress
)
assert len(records) == SUBSET_SIZE
print(f"Completed run {logger.run_id}.")

In [ ]:
from src.baseline.experiment import build_baseline_summary, save_baseline_summary

summary = build_baseline_summary(records)
save_baseline_summary(summary, SUMMARY_PATH)
print(json.dumps(summary, indent=2, ensure_ascii=False))
assert summary["number_of_problems"] == SUBSET_SIZE

In [ ]:
malformed = [record for record in records if record["predicted_answer"] is None and record["is_correct"] is not None]
incorrect = [record for record in records if record["is_correct"] is False]
incomplete = [record for record in records if record["is_correct"] is None]
print(f"Incomplete outputs excluded from accuracy: {len(incomplete)}")
print(f"Malformed outputs: {len(malformed)}")
print(f"Incorrect answers: {len(incorrect)}")
for record in (malformed + incorrect)[:5]:
    print("-" * 80)
    print(record["problem_id"], "reference=", record["reference_answer"], "predicted=", record["predicted_answer"])
    print(record["reasoning_trace"][:2000])

The completed artifacts are `results/baseline/baseline_results.jsonl` and `results/baseline/baseline_summary.json`. Download or copy them to persistent storage before the Colab runtime ends.